# 13.15 一輪選卡與更新，如何接起來？


現在用同一組四卡做一次完整循環。題目「只回數字：1+2=?」的理想卡是0號「3」。輸入小網路的不是中文原文，而是`[0.1,0.2,0,0]`：兩個數各除10、是否要求一句話、是否缺購買數量。後兩格1表示有，0表示沒有。

這個表示把情境直接交給模型；回答內容也由Python預先寫好。它只學哪張卡合適，沒有學中文閱讀、加法計算或逐token生成。這樣縮小任務，是為了能追清PPO中每份數值的去向。

一輪有四步：

1. 目前policy按情境抽一張卡，保存該卡的舊log機率。
2. 固定reward model評卡，更新前critic給基準，兩者相減保存advantage。
3. 用目前policy重算同卡機率，求ratio與裁切代價，另加reference的KL；critic另算分數預測代價。
4. 清舊梯度、`backward()`算新梯度，最後更新器`step()`才改policy與critic。reward與reference不在更新器裡。

下面的完整程式先用隨機小網路核對「誰真的變了」。它的評分員尚未教好，所以參數更新只能證明流程接通，不能證明任務變好。

<details>
<summary>完整小程式：一次收集與一次更新</summary>


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import copy
import torch
from tiny_perceptron.posttraining import (
    FiniteResponsePolicy,
    FiniteRewardModel,
    FiniteValueModel,
    bandit_advantage,
    exact_kl,
    ppo_clipped_objective,
)

torch.manual_seed(42)
context = torch.tensor([[0.1, 0.2, 0.0, 0.0]])
policy, critic = FiniteResponsePolicy(), FiniteValueModel()
reward_model = FiniteRewardModel().requires_grad_(False)
reference = copy.deepcopy(policy).requires_grad_(False)
optimizer = torch.optim.SGD(list(policy.parameters()) + list(critic.parameters()), lr=0.1)

with torch.no_grad():
    old_distribution = torch.distributions.Categorical(logits=policy(context))
    action = old_distribution.sample()
    old_log_probability = old_distribution.log_prob(action)
    reward = reward_model(context).gather(1, action[:, None]).squeeze(1)
    advantage = bandit_advantage(reward, critic(context))

new_logits = policy(context)
new_log_probability = torch.distributions.Categorical(logits=new_logits).log_prob(action)
terms = ppo_clipped_objective(new_log_probability, old_log_probability, advantage)
value_loss = (critic(context) - reward).square().mean()
loss = terms["policy_loss"] + 0.1 * exact_kl(new_logits, reference(context)).mean() + 0.5 * value_loss
before = next(policy.parameters()).detach().clone()
optimizer.zero_grad()
loss.backward()
optimizer.step()
print("選中的卡", action.item(), "更新前ratio", terms["ratio"].item())
print("策略參數改變", not torch.equal(before, next(policy.parameters())))
print("評分員有梯度", any(p.grad is not None for p in reward_model.parameters()))
print("輸入與選卡分數形狀", list(context.shape), list(new_logits.shape))
print("固定參考有梯度", any(p.grad is not None for p in reference.parameters()))

`Categorical`將四張卡分數轉成抽樣分佈，`gather`取已選卡的reward。收集區塊的`no_grad`保存固定紀錄；更新區塊重新算可求梯度的新機率。固定seed42時，選卡0、更新前ratio1，policy改變為True，評分員和reference有梯度都為False。ratio1是尚未做第一次更新的結果，不表示裁切把它鎖住。

</details>

真正保存的選卡實驗先用只回數字示範教起點，再用比較資料教評分員，凍結評分員後進入PPO。題目按兩個數的家族一起分組，交換數字或改要求仍留在同一組；最後18題不參與更新。完整要求由起點6／18，變成PPO後12／18，下一節看哪些條件改變。

練習把情境第三格改1，表示要求一句話。輸入與輸出形狀仍是一題四格，理想卡改成1號。隨機程式不保證選到它；「流程可更新」與「學會按要求選卡」要分別驗收。

<details>
<summary>補充：來源、完整設定與既有實測紀錄</summary>

完整CPU實驗則真的先教選卡與評分，再接PPO。兩個數都在1到10，55個無序加數家族先分44個訓練、5個驗證、6個最後檢查；每家族各有三種條件，分別132、15、18題。只回數字的44筆示範，教148參數的選卡網路60次，每批32題。這一步是從隨機網路開始的監督式示範訓練：給出情境與示範答案，教模型模仿示範的選擇。它借用SFT階段的教法；LLM流程中的SFT（supervised fine-tuning，監督式微調）是在已預訓練模型上接續做示範訓練，見[7.17](https://birdhackor.github.io/tiny-perceptron-vlm/7.17.html)。

這個起點使用0.2的label smoothing，意思是把原本全給正確卡的目標份額稍微攤開。四張卡若以0號為正確答案，原目標是`[1,0,0,0]`；保留其中80%，再把20%平均分給四張卡，得到`[0.85,0.05,0.05,0.05]`。正確卡仍拿最多份額，其他卡也保留少量，避免探索時它們的機率過早接近0。這是訓練時使用的目標分布，不表示模型訓練後必定輸出這四個機率；也不是所有SFT都必須採用的配方。

241參數的評分員用660對訓練偏好，更新300次、每批64對，共抽用19,200對；97參數的critic在PPO階段學分數估計。PPO收集120批，每批64次真實選卡，共7,680次；每批重用三回合，所以策略與critic各更新360次。重用不算新作答，報告另記23,040次被更新流程讀取的動作紀錄。這個選卡實驗沒有token訓練，`effective_tokens=0`有它的單位理由，不表示沒有有效學習資料。

```bash
python -m scripts.course_experiments.run --experiment posttraining --device cpu
```

先依[W.1](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.1)啟用專案Python環境，再在專案根目錄執行。完整實驗含DPO分支，CPU兩個執行緒的這次總耗時約2.12秒；PPO段約0.66秒，未計另一台機器安裝環境的時間。[公開實測報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/results/posttraining.json)保存設定、分組、候選、每題機率、更新紀錄與權重指紋。最後18題的完整要求，起點6題通過，PPO後12題通過；哪些成功、哪些仍失敗，下一節逐項看。

</details>
